what's new in this code: patient-level bags, multi-scale vision feat concatenation

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(r"C:\Users\Vivian\Documents\CONCH")
sys.path.insert(0, str(PROJECT_ROOT))

from conch.open_clip_custom import create_model_from_pretrained, tokenize, get_tokenizer
import torch
import os
from PIL import Image
from pathlib import Path

# show all jupyter output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

c:\Users\Vivian\anaconda3\envs\conch\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# checkpoint_path = 'checkpoints/CONCH/pytorch_model.bin'
checkpoint_path = 'C:\\Users\\Vivian\\Documents\\CONCH\\checkpoints\\conch\\pytorch_model.bin' 
# checkpoint_path = r"C:\Users\Vivian\Documents\CONCH\_finetune_weights\fine_tuned_model2.pth" # testing with finetuned conch model
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
_ = model.eval()

C:\Users\Vivian\Documents\CONCH\conch\open_clip_custom\factory.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=map

In [3]:
# import torch
from transformers import AutoTokenizer
tokenizer = get_tokenizer()

In [4]:
# ============================================================
# (BLOCK 1) SETUP + CONFIG
# - Defines paths, CV splits, and experiment list
# - You will only edit paths + (optionally) FEAT_BACKEND
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

# ---- OPTIONAL: if you later switch UNI gated MIL to .h5, set FEAT_BACKEND="h5" and provide H5_FEAT_DIR
import h5py

# -----------------
# PATHS (EDIT THESE)
# -----------------
# PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\5x\noCAP_patch_ptfile_notopk_conf_PATCH.csv"
# PATCH_CONCEPT_CSV = r"C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\noCAP_patch_ptfile_notopk_conf_PATCH.csv" # 10x
# PATCH_CONCEPT_CSV = r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\2.5x\noCAP_patch_ptfile_notopk_conf_PATCH.csv' # 2.5x
# PATCH_CONCEPT_CSV = r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\data_v2\10x\datav2_concept_prior_PATCH.csv' # 10x v2
# PATCH_CONCEPT_CSV = r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\data_v2\5x\datav2_concept_prior_PATCH.csv' # 2.5x v2
PATCH_CONCEPT_CSV = r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\data_v2\2p5x_norm\datav2_concept_prior_PATCH.csv' # 2p5x v2 norm
# PATCH_CONCEPT_CSV = r'C:\Users\Vivian\Documents\CONCH\test_text_encoder\slide_concept_scores\data_v2\10x_norm_v2\datav2_concept_prior_PATCH.csv' # 10x v2 norm v2

CV_ROOT           = r"C:/Users/Vivian/Documents/PANTHER/PANTHER/src/splits/cross-val"

# Deep features backend for gated experiments (needs coords)
# - "pt": expects {features, coords} in each .pt
# - "h5": expects datasets "features" and "coords" in each .h5
FEAT_BACKEND = "pt"

# PT_FEAT_DIR = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\40x\feats_pt"  # used if FEAT_BACKEND="pt"
# PT_FEAT_DIR = r"C:\Users\Vivian\Documents\CONCH\conch_img_feats\10x_feats\conchextracted_mag10x_patch224_fp\feats_pt"  # used if FEAT_BACKEND="pt" # 10x
# PT_FEAT_DIR = r'C:\Users\Vivian\Documents\CONCH\conch_img_feats\2.5x_feats\40x\pt' #2.5x or 5x
# PT_FEAT_DIRS = [r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\conch\10x",
#     r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\conch\10x",
#                 ]

# v1 data
# PT_FEAT_DIRS = {"2p5x": r"Z:\vivian_patches\miccai_img_feats\conch_img_feats\2.5x_feats\40x\pt",
#                 "5x": r"Z:\vivian_patches\miccai_img_feats\conch_img_feats\5x_feats\40x\pt",
#     "10x": r"Z:\vivian_patches\miccai_img_feats\conch_img_feats\10x_feats\conchextracted_mag10x_patch224_fp\feats_pt"}

# MULTI_SCALE v2 data
# PT_FEAT_DIRS = {
#     "2p5x": [
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\conch\2p5x",
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\conch\2p5x",
#     ],
#     "5x": [
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\conch\5x",
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\conch\5x",
#     ],
#     "10x": [
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\conch\10x",
#         r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\conch\10x",
#     ],
# }

# Virchow2 v2 data
PT_FEAT_DIRS = {
    "2p5x": [
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\virchow2\2p5x",
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\virchow2\2p5x",
    ],
    "5x": [
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\virchow2\5x",
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\virchow2\5x",
    ],
    "10x": [
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\FA\virchow2\10x",
        r"C:\Users\Vivian\Documents\TRIDENT\feat_h5\allFE\multiscene\virchow2\10x",
    ],
}

H5_FEAT_DIR = r""  # used if FEAT_BACKEND="h5" (e.g., UNI feats_h5 folder)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

# For concept-based top-k selection
CONCEPT_SCORE_MODE = "mean_all"   # "mean_all" or "mean_subset"
CONCEPT_SUBSET_IDXS = None        # e.g. [0,1,2] if using subset scoring

# columns in concept CSV
SLIDE_COL = "slide_id"
LABEL_COL = "true_label_str"
X_COL, Y_COL = "x", "y"
CONCEPT_PREFIX = "concept_" # change to variant if using concept variants in CSV

# Bag controls (apply consistently across experiments)
BAG_CAP = None         # e.g., 4096 or None
TOPK    = None         # e.g., 1024 or None
TOPK_MODE = "deep_norm"  # deep_norm or concept_score (only used in gated model dataset)

# Train controls (keep fixed across experiments)
EPOCHS = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
HID = 128
DROPOUT = 0.1

# torch.manual_seed(SEED)
# np.random.seed(SEED)

SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Experiments to run
EXPERIMENTS = [
    "vision_pool",          # pooling (mean) over deep feats
    "vision_abmil",         # ABMIL on deep feats
    # "concept_pool",         # pooling (mean) over concept feats
    # "concept_abmil",        # ABMIL on concept feats
    "vision_concept_gated", # your gated MIL: deep pooled, attention uses deep+concept
]


In [5]:
SCALE_MODE = "10x"   # options: "2p5x", "5x", "10x", "multi"

SAVE_CONCEPT_SCORES = False

# EXPERIMENT_NAME = "RB"
EXPERIMENT_NAME = "RR"
# EXPERIMENT_NAME = "BB"
# EXPERIMENT_NAME = "CB" # combined - biopsy
# EXPERIMENT_NAME = "CR" # combined - resection

BASE_OUT_DIR = (
    r"C:\Users\Vivian\Documents\CONCH\test_text_encoder"
    r"\slide_concept_scores\data_v2\slide_preds_norm_v2" # change to data_v2 if using v2 concept CSV
)
 
OUT_DIR = os.path.join(
    BASE_OUT_DIR,
    EXPERIMENT_NAME,
    SCALE_MODE,
)

os.makedirs(OUT_DIR, exist_ok=True)

In [6]:
# ============================================================
# (BLOCK 2) HELPERS
# - split reading, metrics, safe filtering, fold discovery
# ============================================================

def labels_to_int(arr):
    s = pd.Series(arr).astype(str).str.upper().str.strip()
    y = s.map({"FA": 0, "PT": 1})
    if y.isna().any():
        raise ValueError(f"Bad labels: {s[y.isna()].unique()}")
    return y.astype(int).to_numpy()

def read_slide_list(csv_path: str):
    d = pd.read_csv(csv_path)
    if "slide" in d.columns:
        slides = d["slide"].astype(str).tolist()
    elif "slide_id" in d.columns:
        slides = d["slide_id"].astype(str).tolist()
    else:
        slides = d.iloc[:, 0].astype(str).tolist()
    return [os.path.splitext(s.strip())[0] for s in slides]

def find_fold_dirs(cv_root: str):
    out = []
    for name in sorted(os.listdir(cv_root)):
        p = os.path.join(cv_root, name)
        if os.path.isdir(p) and all(os.path.isfile(os.path.join(p, f"{x}.csv")) for x in ["train","val","test"]):
            out.append(p)
    return out

def eval_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_prob)
    if mask.sum() == 0:
        return {"auc": np.nan, "balacc": np.nan, "acc": np.nan}
    y_true = y_true[mask].astype(int)
    y_prob = y_prob[mask]
    y_pred = (y_prob >= thr).astype(int)
    auc = np.nan
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_prob)
    return {
        "auc": auc,
        "balacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
    }

def summarize_percent(df_metrics, cols=("auc","balacc","acc")):
    out = {}
    for c in cols:
        m = df_metrics[c].mean() * 100.0
        s = df_metrics[c].std(ddof=1) * 100.0
        out[c] = (round(m, 2), round(s, 2))
    return out

# OG using one scale at a time ================================
# added helper to find pt files for multiple dirs
# MAG = "10x" # change for each mag
# ENC = "conch"

# def find_pt_path(slide_id: str):
#     candidates = []

#     for d in PT_FEAT_DIRS:
#         possible = [
#             os.path.join(d, f"{slide_id}.pt"),                    # old style
#             os.path.join(d, f"{slide_id}_{MAG}_{ENC}.pt"),         # new style
#         ]

#         for p in possible:
#             if os.path.isfile(p):
#                 candidates.append(p)

#     if len(candidates) == 0:
#         raise FileNotFoundError(f"No PT file found for {slide_id}")

#     if len(candidates) > 1:
#         print(f"[WARN] Multiple matches for {slide_id}; using {candidates[0]}")

#     return candidates[0]

# using multiple feat scales ============================================
MAG = "10x" # change for each mag
ENC = "virchow2"

def find_pt_path(slide_id: str, mag: str):
    candidates = []

    for d in PT_FEAT_DIRS[mag]:
        p = os.path.join(d, f"{slide_id}_{mag}_{ENC}.pt")

        if os.path.isfile(p):
            candidates.append(p)

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No PT file found for {slide_id} ({mag})"
        )

    if len(candidates) > 1:
        print(f"[WARN] Multiple matches for {slide_id}; using {candidates[0]}")

    return candidates[0]

In [7]:
# new helper to calc patient level metrics
import re
import pandas as pd
import numpy as np

def infer_patient_id(slide_id):
    s = str(slide_id).strip()
    m = re.match(r"^(FA|PT)[ -]?(\d+)", s.upper())
    if not m:
        raise ValueError(f"Could not infer patient from slide_id={slide_id}")
    return f"{m.group(1)} {m.group(2)}"

def patient_level_metrics_from_slide_rows(slide_rows, thr=0.5):
    """
    slide_rows must contain:
      sid, y, prob
    where y: FA=0, PT=1
    prob: predicted PT probability
    """

    df = pd.DataFrame(slide_rows)
    df["patient_id"] = df["sid"].map(infer_patient_id)
    df["slide_pred"] = (df["prob"] >= thr).astype(int)

    # patient true label should be same across slides
    patient_df = (
        df.groupby("patient_id")
          .agg(
              y=("y", "first"),

              # conservative rule: if any slide predicted PT => patient PT
              pred_any_pt=("slide_pred", "max"),

              # useful for AUC
              prob_max=("prob", "max"),
              prob_mean=("prob", "mean"),

              n_slides=("sid", "count"),
              slides=("sid", lambda x: "; ".join(x)),
          )
          .reset_index()
    )

    # choose one probability for AUC
    # prob_max matches "if one slide is PT, call patient PT" better
    m = eval_metrics(
        patient_df["y"].to_numpy(),
        patient_df["prob_max"].to_numpy(),
        thr=thr,
    )

    # overwrite ACC/BAC using hard any-PT rule
    m["acc"] = accuracy_score(patient_df["y"], patient_df["pred_any_pt"])
    m["balacc"] = balanced_accuracy_score(patient_df["y"], patient_df["pred_any_pt"])

    return m, patient_df

In [8]:
# ============================================================
# (BLOCK 3) DEEP FEATURE LOADERS (WITH COORDS)
# - pt backend: expects dict with keys {"features","coords"}
# - h5 backend: expects datasets {"features","coords"} (CLAM style)
# ============================================================

def load_slide_deep_with_coords(slide_id: str, mag: str): # added mag for multi-scale support
    if FEAT_BACKEND == "pt":

        #================ og
        # pt_path = os.path.join(PT_FEAT_DIR, f"{slide_id}.pt")
        # if not os.path.isfile(pt_path):
        #     raise FileNotFoundError(pt_path)
        #================ og

        # pt_path = find_pt_path(slide_id) # for multiple PT_FEAT_DIRS support OG
        pt_path = find_pt_path(slide_id, mag) # for multi-scale support

        obj = torch.load(pt_path, map_location="cpu", weights_only=True)  # don't use weights_only here # trying with weights only
        if not isinstance(obj, dict) or "features" not in obj or "coords" not in obj:
            raise ValueError(f"Expected dict with keys 'features','coords' in {pt_path}. Got type={type(obj)} keys={getattr(obj,'keys',lambda:[])()}")
        feats = obj["features"].float().numpy()
        coords = obj["coords"].int().numpy()
        return feats.astype(np.float32), coords.astype(np.int32)

    elif FEAT_BACKEND == "h5":
        if not H5_FEAT_DIR:
            raise ValueError("Set H5_FEAT_DIR when FEAT_BACKEND='h5'")
        h5_path = os.path.join(H5_FEAT_DIR, f"{slide_id}.h5")
        if not os.path.isfile(h5_path):
            raise FileNotFoundError(h5_path)
        with h5py.File(h5_path, "r") as f:
            if "features" in f:
                feats = f["features"][:]
            elif "feats" in f:
                feats = f["feats"][:]
            else:
                raise KeyError(f"No 'features'/'feats' in {h5_path}. Keys={list(f.keys())}")
            if "coords" not in f:
                raise KeyError(f"No 'coords' in {h5_path}. Keys={list(f.keys())}")
            coords = f["coords"][:]
        return feats.astype(np.float32), coords.astype(np.int32)

    else:
        raise ValueError(f"Bad FEAT_BACKEND='{FEAT_BACKEND}'")


In [9]:
# # added this for multiscale feature support
# def load_multiscale_features(slide_id):

#     X25, coords = load_slide_deep_with_coords(slide_id, "2p5x")
#     X5, _       = load_slide_deep_with_coords(slide_id, "5x")
#     X10, _      = load_slide_deep_with_coords(slide_id, "10x")

#     X = np.concatenate([X25, X5, X10], axis=1)

#     return X.astype(np.float32), coords

# SCALE_MODE = "multi"   # options: "2p5x", "5x", "10x", "multi"

def load_multiscale_features(slide_id):
    if SCALE_MODE == "multi":
        X25, coords = load_slide_deep_with_coords(slide_id, "2p5x")
        X5, _       = load_slide_deep_with_coords(slide_id, "5x")
        X10, _      = load_slide_deep_with_coords(slide_id, "10x")
        X = np.concatenate([X25, X5, X10], axis=1)
        return X.astype(np.float32), coords

    else:
        X, coords = load_slide_deep_with_coords(slide_id, SCALE_MODE)
        return X.astype(np.float32), coords

In [10]:
# (BLOCK 4) LOAD CONCEPT CSV + PREP CONCEPT COLUMNS
# ============================================================

df_patch = pd.read_csv(PATCH_CONCEPT_CSV) 

df_patch.columns = [c.strip() for c in df_patch.columns]

# added to normalize slide_id ============== datav2
# df_patch[SLIDE_COL] = (
#     df_patch[SLIDE_COL]
#     .astype(str)
#     .str.replace(f"_{MAG}_{ENC}", "", regex=False)
#     .str.strip()
# )

# for use with virchow and uni feats
df_patch[SLIDE_COL] = (
    df_patch[SLIDE_COL]
    .astype(str)
    .str.replace(
        r"_(2p5x|5x|10x)_(conch|virchow2|uni|uni2)$",
        "",
        regex=True,
    )
    .str.strip()
)
# =========================================

for col in [SLIDE_COL, LABEL_COL, X_COL, Y_COL]:
    if col not in df_patch.columns:
        raise ValueError(f"Missing required column '{col}' in PATCH_CONCEPT_CSV")

concept_cols = [c for c in df_patch.columns if c.startswith(CONCEPT_PREFIX)]
if not concept_cols:
    raise ValueError("No concept_* columns found in PATCH_CONCEPT_CSV")

df_patch[concept_cols] = df_patch[concept_cols].apply(pd.to_numeric, errors="coerce")
df_patch[concept_cols] = df_patch[concept_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# label map by slide (from your CSV)
lab = df_patch.groupby(SLIDE_COL)[LABEL_COL].first()
y_map_global = {sid: int(labels_to_int([lab.loc[sid]])[0]) for sid in lab.index.tolist()}


In [12]:
# (BLOCK 5) DATASETS
# A) VisionOnlyBagDataset: deep features only (no coords needed for model, but loader provides coords anyway)
# B) ConceptOnlyBagDataset: concept vectors aligned to deep coords (so instances match your deep patches)
# C) VisionConceptBagDataset: returns (Xd, Xc) aligned
# ============================================================

class VisionOnlyBagDataset(Dataset):
    def __init__(self, slide_ids, bag_cap=None, topk=None, seed=0):
        self.slide_ids = [s for s in slide_ids if s in y_map_global]
        self.bag_cap = bag_cap
        self.topk = topk
        self.rng = np.random.default_rng(seed)

        # filter to slides with deep feature files
        keep = []
        for sid in self.slide_ids:
            try:
                # _ = load_slide_deep_with_coords(sid)
                _ = load_multiscale_features(sid) # for multi-scale support
                keep.append(sid)
            except Exception:
                pass
        self.slide_ids = keep

    def __len__(self): return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        # Xd, _coords = load_slide_deep_with_coords(sid)  # (N,D)
        Xd, coords = load_multiscale_features(sid)
        N = Xd.shape[0]

        sel = np.arange(N)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)
            Xd = Xd[sel]

        if self.topk is not None and Xd.shape[0] > self.topk:
            score = np.linalg.norm(Xd, axis=1)
            keep = np.argsort(-score)[: self.topk]
            Xd = Xd[keep]

        y = y_map_global[sid]
        return torch.from_numpy(Xd), torch.tensor(y, dtype=torch.long), sid


class ConceptOnlyBagDataset(Dataset):
    """
    Concept-only MIL needs a set of instances per slide.
    We make instances match the deep patch grid by aligning concept(x,y) to deep coords.
    """
    def __init__(self, df_patch_concept, slide_ids, concept_cols, bag_cap=None, topk=None,
                 topk_mode="deep_norm", concept_score_mode="mean_all", concept_subset_idxs=None, seed=0):
        self.df = df_patch_concept[df_patch_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique()) and s in y_map_global]
        self.concept_cols = concept_cols

        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode
        self.concept_score_mode = concept_score_mode
        self.concept_subset_idxs = concept_subset_idxs

        self.rng = np.random.default_rng(seed)
        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}

        # also filter to slides with deep coords available (needed for alignment)
        keep = []
        for sid in self.slide_ids:
            try:
                # _ = load_slide_deep_with_coords(sid)
                _ = load_multiscale_features(sid) # for multi-scale support
                keep.append(sid)
            except Exception:
                pass
        self.slide_ids = keep

    def __len__(self): return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        # _Xd, coords = load_slide_deep_with_coords(sid)
        Xd, coords = load_multiscale_features(sid) # for multi-scale support
        d = self.by_slide[sid]

        # concept map (x,y)->vec
        keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(keys)}

        # fill Xc
        Xc = np.zeros((coords.shape[0], len(self.concept_cols)), dtype=np.float32)
        for i, (x, y) in enumerate(coords.astype(int)):
            v = concept_map.get((int(x), int(y)))
            if v is not None:
                Xc[i] = v

        # selection (cap) on concepts only
        N = Xc.shape[0]
        sel = np.arange(N)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)
            Xc = Xc[sel]

        # # OPTIONAL: z-score per slide (after selection)
        # mu = Xc.mean(axis=0, keepdims=True)
        # sd = Xc.std(axis=0, keepdims=True) + 1e-6
        # Xc = (Xc - mu) / sd

        y = y_map_global[sid]
        return torch.from_numpy(Xc), torch.tensor(y, dtype=torch.long), sid



class VisionConceptBagDataset(Dataset):
    """
    Returns aligned (Xd, Xc) by deep coords.
    """
    def __init__(self, df_patch_concept, slide_ids, concept_cols, bag_cap=None, topk=None,
                 topk_mode="deep_norm", concept_score_mode="mean_all", concept_subset_idxs=None, seed=0):
        self.df = df_patch_concept[df_patch_concept[SLIDE_COL].isin(slide_ids)].copy()
        self.slide_ids = [s for s in slide_ids if s in set(self.df[SLIDE_COL].unique()) and s in y_map_global]
        self.concept_cols = concept_cols

        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode
        self.concept_score_mode = concept_score_mode
        self.concept_subset_idxs = concept_subset_idxs

        self.rng = np.random.default_rng(seed)
        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}

        # filter to slides with deep feats
        keep = []
        for sid in self.slide_ids:
            try:
                # _ = load_slide_deep_with_coords(sid)
                _ = load_multiscale_features(sid) # for multi-scale support
                keep.append(sid)
            except Exception:
                pass
        self.slide_ids = keep

    def __len__(self): return len(self.slide_ids)


    # saving coords for
    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        # Xd, coords = load_slide_deep_with_coords(sid)  # coords (N,2)
        Xd, coords = load_multiscale_features(sid) # for multi-scale support
        d = self.by_slide[sid]

        keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
        C = d[self.concept_cols].to_numpy(dtype=np.float32)
        concept_map = {k: C[i] for i, k in enumerate(keys)}

        Xc = np.zeros((coords.shape[0], len(self.concept_cols)), dtype=np.float32)
        for i, (x, y) in enumerate(coords.astype(int)):
            v = concept_map.get((int(x), int(y)))
            if v is not None:
                Xc[i] = v

        # selection
        N = Xd.shape[0]
        sel = np.arange(N)
        if self.bag_cap is not None and N > self.bag_cap:
            sel = self.rng.choice(N, size=self.bag_cap, replace=False)

        Xd = Xd[sel].astype(np.float32)
        Xc = Xc[sel].astype(np.float32)
        coords_sel = coords[sel].astype(np.int32)

        if self.topk is not None and Xd.shape[0] > self.topk:
            if self.topk_mode == "deep_norm":
                score = np.linalg.norm(Xd, axis=1)
            elif self.topk_mode == "concept_score":
                score = Xc.mean(axis=1)
            keep = np.argsort(-score)[: self.topk]
            Xd = Xd[keep]
            Xc = Xc[keep]
            coords_sel = coords_sel[keep]

        y = y_map_global[sid]
        return torch.from_numpy(Xd), torch.from_numpy(Xc), torch.tensor(y, dtype=torch.long), sid, torch.from_numpy(coords_sel)


def collate_bag(batch):
    return batch  # list of length B, each is one slide


In [13]:
# NEW patient-level bags
def make_patient_items(slide_ids):
    patient_to_slides = {}

    for sid in slide_ids:
        if sid not in y_map_global:
            continue

        pid = infer_patient_id(sid)
        patient_to_slides.setdefault(pid, []).append(sid)

    return sorted(patient_to_slides.items())


class PatientVisionOnlyBagDataset(Dataset):
    def __init__(self, slide_ids, bag_cap=None, topk=None, seed=0):
        self.items = make_patient_items(slide_ids)
        self.bag_cap = bag_cap
        self.topk = topk
        self.rng = np.random.default_rng(seed)

        keep = []
        for pid, sids in self.items:
            valid = []
            for sid in sids:
                try:
                    # _ = load_slide_deep_with_coords(sid)
                    _ = load_multiscale_features(sid) # for multi-scale support
                    valid.append(sid)
                except Exception:
                    pass
            if valid:
                keep.append((pid, valid))

        self.items = keep

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, sids = self.items[idx]

        X_all = []
        ys = []

        for sid in sids:
            # Xd, _coords = load_slide_deep_with_coords(sid)
            Xd, _ = load_multiscale_features(sid) # for multi-scale support
            X_all.append(Xd)
            ys.append(y_map_global[sid])

        Xd = np.concatenate(X_all, axis=0).astype(np.float32)
        y = int(max(ys))

        if self.bag_cap is not None and Xd.shape[0] > self.bag_cap:
            sel = self.rng.choice(Xd.shape[0], size=self.bag_cap, replace=False)
            Xd = Xd[sel]

        if self.topk is not None and Xd.shape[0] > self.topk:
            score = np.linalg.norm(Xd, axis=1)
            keep = np.argsort(-score)[:self.topk]
            Xd = Xd[keep]

        return torch.from_numpy(Xd), torch.tensor(y, dtype=torch.long), pid


class PatientVisionConceptBagDataset(Dataset):
    def __init__(self, df_patch_concept, slide_ids, concept_cols, bag_cap=None, topk=None,
                 topk_mode="deep_norm", seed=0):
        self.df = df_patch_concept.copy()
        self.concept_cols = concept_cols
        self.bag_cap = bag_cap
        self.topk = topk
        self.topk_mode = topk_mode
        self.rng = np.random.default_rng(seed)

        self.by_slide = {sid: d for sid, d in self.df.groupby(SLIDE_COL)}
        self.items = make_patient_items(slide_ids)

        keep = []
        for pid, sids in self.items:
            valid = []
            for sid in sids:
                try:
                    # _ = load_slide_deep_with_coords(sid)
                    _ = load_multiscale_features(sid) # for multi-scale support
                    if sid in self.by_slide:
                        valid.append(sid)
                except Exception:
                    pass
            # print(pid, len(valid), valid[:3])
            if valid:
                keep.append((pid, valid))

        self.items = keep

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, sids = self.items[idx]

        Xd_all = []
        Xc_all = []
        coords_all = []
        ys = []

        for sid in sids:
            # Xd, coords = load_slide_deep_with_coords(sid)
            Xd, coords = load_multiscale_features(sid) # for multi-scale support
            d = self.by_slide[sid]

            keys = list(zip(d[X_COL].astype(int).to_numpy(), d[Y_COL].astype(int).to_numpy()))
            C = d[self.concept_cols].to_numpy(dtype=np.float32)
            concept_map = {k: C[i] for i, k in enumerate(keys)}

            Xc = np.zeros((coords.shape[0], len(self.concept_cols)), dtype=np.float32)
            for i, (x, ycoord) in enumerate(coords.astype(int)):
                v = concept_map.get((int(x), int(ycoord)))
                if v is not None:
                    Xc[i] = v

            Xd_all.append(Xd)
            Xc_all.append(Xc)
            coords_all.append(coords)
            ys.append(y_map_global[sid])

        Xd = np.concatenate(Xd_all, axis=0).astype(np.float32)
        Xc = np.concatenate(Xc_all, axis=0).astype(np.float32)
        coords = np.concatenate(coords_all, axis=0).astype(np.int32)
        y = int(max(ys))

        if self.bag_cap is not None and Xd.shape[0] > self.bag_cap:
            sel = self.rng.choice(Xd.shape[0], size=self.bag_cap, replace=False)
            Xd = Xd[sel]
            Xc = Xc[sel]
            coords = coords[sel]

        if self.topk is not None and Xd.shape[0] > self.topk:
            score = np.linalg.norm(Xd, axis=1)
            keep = np.argsort(-score)[:self.topk]
            Xd = Xd[keep]
            Xc = Xc[keep]
            coords = coords[keep]

        return (
            torch.from_numpy(Xd),
            torch.from_numpy(Xc),
            torch.tensor(y, dtype=torch.long),
            pid,
            torch.from_numpy(coords),
        )

In [14]:
# (BLOCK 6) MODELS
# A) PoolingClassifier: mean pooling then linear classifier
# B) AttentionMIL: ABMIL (single-stream)
# C) GatedAttentionMIL: your deep pooled, attention uses deep+concept
# ============================================================

class PoolingClassifier(nn.Module):
    def __init__(self, in_dim, hid=128, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hid, 1),
        )

    def forward(self, X):  # X:(N,D)
        z = X.mean(dim=0)  # (D,)
        logit = self.net(z).squeeze(0)
        return logit


class AttentionMIL(nn.Module):
    def __init__(self, in_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(drop),
        )
        self.attn = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    def forward(self, X):  # X:(N,D)
        H = self.phi(X)                 # (N,hid)
        a = self.attn(H).squeeze(1)     # (N,)
        w = torch.softmax(a, dim=0)     # (N,)
        z = (w.unsqueeze(1) * H).sum(0) # (hid,)
        logit = self.cls(z).squeeze(0)
        return logit, w


class GatedAttentionMIL(nn.Module):
    """
    Attention logits: a_i = attn_d(phi_d(Xd_i)) + attn_c(phi_c(Xc_i))
    Pooling uses deep stream only (Hd).
    """
    def __init__(self, deep_dim, concept_dim, hid=128, drop=0.1):
        super().__init__()
        self.phi_d = nn.Sequential(nn.Linear(deep_dim, hid), nn.ReLU(), nn.Dropout(drop))
        self.phi_c = nn.Sequential(nn.Linear(concept_dim, hid), nn.ReLU(), nn.Dropout(drop))
        self.attn_d = nn.Linear(hid, 1)
        self.attn_c = nn.Linear(hid, 1)
        self.cls = nn.Linear(hid, 1)

    # def forward(self, Xd, Xc):
    #     Hd = self.phi_d(Xd)
    #     Hc = self.phi_c(Xc)
    #     a = self.attn_d(Hd).squeeze(1) + self.attn_c(Hc).squeeze(1)
    #     w = torch.softmax(a, dim=0)
    #     z = (w.unsqueeze(1) * Hd).sum(0)
    #     logit = self.cls(z).squeeze(0)
    #     return logit, w
    
    # coords
    def forward(self, Xd, Xc, return_parts=False):
        Hd = self.phi_d(Xd)
        Hc = self.phi_c(Xc)
        a_d = self.attn_d(Hd).squeeze(1)
        a_c = self.attn_c(Hc).squeeze(1)
        a = a_d + a_c
        w = torch.softmax(a, dim=0)
        z = (w.unsqueeze(1) * Hd).sum(0)
        logit = self.cls(z).squeeze(0)
        if return_parts:
            return logit, w, a_d, a_c
        return logit, w


In [14]:
# function to save concept scores for each slide (for later use in gated MIL)
def save_slide_concepts(
    sid,
    y,
    prob,
    coords,
    Xc,
    attn,
    fold_name,
    out_dir,
):
    w = attn.detach().cpu().numpy().astype(np.float32)
    Xc = Xc.detach().cpu().numpy().astype(np.float32)

    np.savez_compressed(
        os.path.join(out_dir, f"{sid}.npz"),

        slide_id=sid,
        y_true=int(y.item()),
        p_pt=prob,

        fold=fold_name,
        scale=SCALE_MODE,
        experiment=EXPERIMENT_NAME,

        coords=coords.numpy().astype(np.int32),
        attn=w,
        Xc=Xc,

        concept_mean=Xc.mean(axis=0).astype(np.float32),
        concept_attn_weighted=(w[:, None] * Xc).sum(axis=0).astype(np.float32),
    )

In [15]:
# (BLOCK 7) TRAIN + EVAL (one fold)
# - Shared training loop for all experiments
# ============================================================

# def train_one_fold(model, dl_tr, dl_te, mode: str): # OG
def train_one_fold(model, dl_tr, dl_te, mode, fold_name):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    for ep in range(EPOCHS):
        for batch in dl_tr:
            item = batch[0]

            if mode == "vision_pool":
                (Xd, y, sid) = item
                Xd = Xd.to(DEVICE); y = y.float().to(DEVICE)
                logit = model(Xd)
            elif mode == "vision_abmil":
                (Xd, y, sid) = item
                Xd = Xd.to(DEVICE); y = y.float().to(DEVICE)
                logit, _ = model(Xd)
            elif mode == "concept_pool":
                (Xc, y, sid) = item
                Xc = Xc.to(DEVICE); y = y.float().to(DEVICE)
                logit = model(Xc)
            elif mode == "concept_abmil":
                (Xc, y, sid) = item
                Xc = Xc.to(DEVICE); y = y.float().to(DEVICE)
                logit, _ = model(Xc)
            elif mode == "vision_concept_gated":
                # (Xd, Xc, y, sid) = item
                (Xd, Xc, y, sid, coords) = item # for coords
                Xd = Xd.to(DEVICE); Xc = Xc.to(DEVICE); y = y.float().to(DEVICE)
                logit, _ = model(Xd, Xc)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            loss = loss_fn(logit.view(1), y.view(1))
            opt.zero_grad()
            loss.backward()
            opt.step()

    # eval
    model.eval()
    y_true, y_prob = [], []
    slide_rows = [] # for patient-level metrics
    with torch.no_grad():
        for batch in dl_te:
            item = batch[0]

            if mode == "vision_pool":
                (Xd, y, sid) = item
                Xd = Xd.to(DEVICE)
                logit = model(Xd)
            elif mode == "vision_abmil":
                (Xd, y, sid) = item
                Xd = Xd.to(DEVICE)
                logit, _ = model(Xd)
            elif mode == "concept_pool":
                (Xc, y, sid) = item
                Xc = Xc.to(DEVICE)
                logit = model(Xc)
            elif mode == "concept_abmil":
                (Xc, y, sid) = item
                Xc = Xc.to(DEVICE)
                logit, _ = model(Xc)
            elif mode == "vision_concept_gated":
                # (Xd, Xc, y, sid) = item
                (Xd, Xc, y, sid, coords) = item # coords
                Xd = Xd.to(DEVICE); Xc = Xc.to(DEVICE)
                logit, w = model(Xd, Xc)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            # OG =============================================
            prob = torch.sigmoid(logit.detach().float().cpu()).item()
           
            # ================================================ save concept scores for gated MIL
            if SAVE_CONCEPT_SCORES and mode == "vision_concept_gated":
                save_slide_concepts(
                    sid,
                    y,
                    prob,
                    coords,
                    Xc,
                    w,
                    fold_name,
                    OUT_DIR,
                )
            # =============================================== save concept scores for gated MIL

            y_true.append(int(y.item()))
            y_prob.append(prob)


    return eval_metrics(np.array(y_true), np.array(y_prob), thr=0.5) # OG
            # ===============================================

    # for patient-level metrics===========================
    #         prob = torch.sigmoid(logit.detach().float().cpu()).item()
    #         y_int = int(y.item())

    #         y_true.append(y_int)
    #         y_prob.append(prob)

    #         slide_rows.append({
    #             "sid": sid,
    #             "y": y_int,
    #             "prob": prob,
    #         })

    # slide_metrics = eval_metrics(np.array(y_true), np.array(y_prob), thr=0.5)
    # patient_metrics, patient_df = patient_level_metrics_from_slide_rows(slide_rows, thr=0.5)

    # return slide_metrics, patient_metrics, patient_df
#=============================================================

In [16]:
# (BLOCK 8) RUN ALL EXPERIMENTS ACROSS 5-FOLD CV
# - Produces a single results table you can paste into paper
# ============================================================
# BAG_CAP = 4096 

fold_dirs = find_fold_dirs(CV_ROOT)

all_rows = []

for mode in EXPERIMENTS:
    print(f"\n==================== {mode} ====================")
    fold_metrics = []

    # for fold_dir in fold_dirs:

# ========================
    for fold_idx, fold_dir in enumerate(fold_dirs):

        fold_seed = SEED + fold_idx

        torch.manual_seed(fold_seed)
        np.random.seed(fold_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(fold_seed)
            torch.cuda.manual_seed_all(fold_seed)
# =============================================================

        fold_name = os.path.basename(fold_dir)
        train_slides = read_slide_list(os.path.join(fold_dir, "train.csv"))
        val_slides   = read_slide_list(os.path.join(fold_dir, "val.csv"))
        test_slides  = read_slide_list(os.path.join(fold_dir, "test.csv")) # change test set here
        fit_slides = train_slides + val_slides

        # debugging
        # print(len(fit_slides))
        # print(len(test_slides))

        # datasets
        if mode in ["vision_pool", "vision_abmil"]:
            ds_tr = VisionOnlyBagDataset(fit_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)
            ds_te = VisionOnlyBagDataset(test_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)

            # patient-level bags
            # ds_tr = PatientVisionOnlyBagDataset(fit_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)
            # ds_te = PatientVisionOnlyBagDataset(test_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)

            # debugging
            # print(f"{fold_name}")
            # print("fit slides:", len(fit_slides))
            # print("test slides:", len(test_slides))

            # print("patient train bags:", len(ds_tr))
            # print("patient test bags:", len(ds_te))

            # debug 
            # print(ds_tr.slide_ids[:5])
            # print(len(ds_tr.slide_ids))

        elif mode in ["concept_pool", "concept_abmil"]:
            ds_tr = ConceptOnlyBagDataset(df_patch, fit_slides, concept_cols,
                                          bag_cap=BAG_CAP, topk=TOPK,
                                          topk_mode=TOPK_MODE,
                                          concept_score_mode=CONCEPT_SCORE_MODE,
                                          concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                          seed=SEED)
            ds_te = ConceptOnlyBagDataset(df_patch, test_slides, concept_cols,
                                          bag_cap=BAG_CAP, topk=TOPK,
                                          topk_mode=TOPK_MODE,
                                          concept_score_mode=CONCEPT_SCORE_MODE,
                                          concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                          seed=SEED)
        elif mode == "vision_concept_gated":
            ds_tr = VisionConceptBagDataset(df_patch, fit_slides, concept_cols,
                                            bag_cap=BAG_CAP, topk=TOPK,
                                            topk_mode=TOPK_MODE,
                                            concept_score_mode=CONCEPT_SCORE_MODE,
                                            concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                            seed=SEED)
            ds_te = VisionConceptBagDataset(df_patch, test_slides, concept_cols,
                                            bag_cap=BAG_CAP, topk=TOPK,
                                            topk_mode=TOPK_MODE,
                                            concept_score_mode=CONCEPT_SCORE_MODE,
                                            concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                            seed=SEED)
            
            # debug
            # print(ds_tr.slide_ids[:5])
            # print(len(ds_tr.slide_ids))
           
            # patient-level bags
        #     ds_tr = PatientVisionConceptBagDataset(
        #     df_patch, fit_slides, concept_cols,
        #     bag_cap=BAG_CAP, topk=TOPK,
        #     topk_mode=TOPK_MODE, seed=SEED
        # )

        #     ds_te = PatientVisionConceptBagDataset(
        #         df_patch, test_slides, concept_cols,
        #         bag_cap=BAG_CAP, topk=TOPK,
        #         topk_mode=TOPK_MODE, seed=SEED
        #     )

        else:
            raise ValueError(mode)

        if len(ds_tr) == 0 or len(ds_te) == 0:
            print(f"[{fold_name}] skipped (empty train/test after filtering)")
            continue


        # reproducible shuffling ================================
        g = torch.Generator()
        g.manual_seed(fold_seed);

        dl_tr = DataLoader(
            ds_tr,
            batch_size=1,
            shuffle=True,
            collate_fn=collate_bag,
            generator=g,
        )
        # ==============================
        # dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
        dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

        # reset before model init ==============
        torch.manual_seed(fold_seed)
        np.random.seed(fold_seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(fold_seed)
            torch.cuda.manual_seed_all(fold_seed)
        # =============================================

        # init model dims
        if mode == "vision_pool":
            Xd0, _, _ = ds_tr[0]
            model = PoolingClassifier(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "vision_abmil":
            Xd0, _, _ = ds_tr[0]
            model = AttentionMIL(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "concept_pool":
            Xc0, _, _ = ds_tr[0]
            model = PoolingClassifier(in_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "concept_abmil":
            Xc0, _, _ = ds_tr[0]
            model = AttentionMIL(in_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "vision_concept_gated":
            # Xd0, Xc0, _, _ = ds_tr[0]
            Xd0, Xc0, y0, sid0, coords0 = ds_tr[0] # with coords
            model = GatedAttentionMIL(deep_dim=Xd0.shape[1], concept_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        # with patient-level metrics =============================
    #     m_slide, m_patient, patient_df = train_one_fold(model, dl_tr, dl_te, mode) 
    #     fold_metrics.append({
    #     "fold": fold_name,
    #     **m_slide,
    #     "level": "slide",
    #     "n_test": len(ds_te),
    # })

    #     fold_metrics.append({
    #         "fold": fold_name,
    #         **m_patient,
    #         "level": "patient",
    #         "n_test": len(patient_df),
    #     })
        # ====================================================

        # OG slide-level ============================================
        # m = train_one_fold(model, dl_tr, dl_te, mode) # OG
        m = train_one_fold(
            model,
            dl_tr,
            dl_te,
            mode,
            fold_name,
        )

        fold_metrics.append({"fold": fold_name, **m, "n_test": len(ds_te)})
        print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(ds_te)})")
        # OG ============================================

    dfm = pd.DataFrame(fold_metrics)
    if len(dfm) == 0:
        print(f"No folds ran for {mode}")
        continue

    summary = summarize_percent(dfm, cols=("auc","balacc","acc"))
    all_rows.append({
        "experiment": mode,
        "AUC (mean±std %)": f"{summary['auc'][0]:.2f} ± {summary['auc'][1]:.2f}",
        "BAC (mean±std %)": f"{summary['balacc'][0]:.2f} ± {summary['balacc'][1]:.2f}",
        "ACC (mean±std %)": f"{summary['acc'][0]:.2f} ± {summary['acc'][1]:.2f}",
        "n_folds": len(dfm),
    })
    

results_table = pd.DataFrame(all_rows)
print("\n==================== FINAL SUMMARY TABLE ====================")
print(results_table.to_string(index=False))



==================== vision_pool ====================


[FA_PT_k=0] auc=0.804 balacc=0.795 acc=0.809 (n_test=47)


[FA_PT_k=1] auc=0.837 balacc=0.699 acc=0.708 (n_test=48)


[FA_PT_k=2] auc=0.802 balacc=0.565 acc=0.600 (n_test=45)


[FA_PT_k=3] auc=0.813 balacc=0.731 acc=0.735 (n_test=49)


...

[FA_PT_k=0] auc=0.851 balacc=0.727 acc=0.745 (n_test=47)


[FA_PT_k=1] auc=0.798 balacc=0.681 acc=0.688 (n_test=48)


[FA_PT_k=2] auc=0.834 balacc=0.585 acc=0.622 (n_test=45)


[FA_PT_k=3] auc=0.922 balacc=0.708 acc=0.714 (n_test=49)


[FA_PT_k=4] auc=0.740 balacc=0.580 acc=0.588 (n_test=51)

==================== FINAL SUMMARY TABLE ====================
          experiment AUC (mean±std %) BAC (mean±std %) ACC (mean±std %)  n_folds
         vision_pool     81.45 ± 1.38     71.05 ± 8.89     72.32 ± 7.83        5
        vision_abmil     82.47 ± 5.03     73.73 ± 5.41     74.32 ± 5.66        5
vision_concept_gated     82.90 ± 6.70     65.63 ± 6.94     67.14 ± 6.48        5


In [ ]:
# (BLOCK 8) RUN ALL EXPERIMENTS ACROSS 5-FOLD CV
# - Produces a single results table you can paste into paper
# PATIENT=LEVEL METRICS aggregated post-hoc
# ============================================================

fold_dirs = find_fold_dirs(CV_ROOT)

all_rows = []

for mode in EXPERIMENTS:
    print(f"\n==================== {mode} ====================")
    fold_metrics = []

    for fold_dir in fold_dirs:
        fold_name = os.path.basename(fold_dir)
        train_slides = read_slide_list(os.path.join(fold_dir, "train_biopsy.csv"))
        val_slides   = read_slide_list(os.path.join(fold_dir, "val_biopsy.csv"))
        test_slides  = read_slide_list(os.path.join(fold_dir, "test_biopsy.csv")) # change test set here
        fit_slides = train_slides + val_slides

        # datasets
        if mode in ["vision_pool", "vision_abmil"]:
            # ds_tr = VisionOnlyBagDataset(fit_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)
            # ds_te = VisionOnlyBagDataset(test_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)

            # patient-level bags
            ds_tr = PatientVisionOnlyBagDataset(fit_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)
            ds_te = PatientVisionOnlyBagDataset(test_slides, bag_cap=BAG_CAP, topk=TOPK, seed=SEED)
        
        elif mode in ["concept_pool", "concept_abmil"]:
            ds_tr = ConceptOnlyBagDataset(df_patch, fit_slides, concept_cols,
                                          bag_cap=BAG_CAP, topk=TOPK,
                                          topk_mode=TOPK_MODE,
                                          concept_score_mode=CONCEPT_SCORE_MODE,
                                          concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                          seed=SEED)
            ds_te = ConceptOnlyBagDataset(df_patch, test_slides, concept_cols,
                                          bag_cap=BAG_CAP, topk=TOPK,
                                          topk_mode=TOPK_MODE,
                                          concept_score_mode=CONCEPT_SCORE_MODE,
                                          concept_subset_idxs=CONCEPT_SUBSET_IDXS,
                                          seed=SEED)
        elif mode == "vision_concept_gated":
            # ds_tr = VisionConceptBagDataset(df_patch, fit_slides, concept_cols,
            #                                 bag_cap=BAG_CAP, topk=TOPK,
            #                                 topk_mode=TOPK_MODE,
            #                                 concept_score_mode=CONCEPT_SCORE_MODE,
            #                                 concept_subset_idxs=CONCEPT_SUBSET_IDXS,
            #                                 seed=SEED)
            # ds_te = VisionConceptBagDataset(df_patch, test_slides, concept_cols,
            #                                 bag_cap=BAG_CAP, topk=TOPK,
            #                                 topk_mode=TOPK_MODE,
            #                                 concept_score_mode=CONCEPT_SCORE_MODE,
            #                                 concept_subset_idxs=CONCEPT_SUBSET_IDXS,
            #                                 seed=SEED)
            
            # patient-level bags
            ds_tr = PatientVisionConceptBagDataset(df_patch, fit_slides, concept_cols,
                                       bag_cap=BAG_CAP, topk=TOPK,
                                       topk_mode=TOPK_MODE, seed=SEED)

            ds_te = PatientVisionConceptBagDataset(df_patch, test_slides, concept_cols,
                                                bag_cap=BAG_CAP, topk=TOPK,
                                                topk_mode=TOPK_MODE, seed=SEED)
        else:
            raise ValueError(mode)

        if len(ds_tr) == 0 or len(ds_te) == 0:
            print(f"[{fold_name}] skipped (empty train/test after filtering)")
            continue

        dl_tr = DataLoader(ds_tr, batch_size=1, shuffle=True, collate_fn=collate_bag)
        dl_te = DataLoader(ds_te, batch_size=1, shuffle=False, collate_fn=collate_bag)

        # init model dims
        if mode == "vision_pool":
            Xd0, _, _ = ds_tr[0]
            model = PoolingClassifier(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "vision_abmil":
            Xd0, _, _ = ds_tr[0]
            model = AttentionMIL(in_dim=Xd0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "concept_pool":
            Xc0, _, _ = ds_tr[0]
            model = PoolingClassifier(in_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "concept_abmil":
            Xc0, _, _ = ds_tr[0]
            model = AttentionMIL(in_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        elif mode == "vision_concept_gated":
            # Xd0, Xc0, _, _ = ds_tr[0]
            Xd0, Xc0, y0, sid0, coords0 = ds_tr[0] # with coords
            model = GatedAttentionMIL(deep_dim=Xd0.shape[1], concept_dim=Xc0.shape[1], hid=HID, drop=DROPOUT).to(DEVICE)

        # with patient-level metrics =============================
        m_slide, m_patient, patient_df = train_one_fold(model, dl_tr, dl_te, mode) 
        fold_metrics.append({
        "fold": fold_name,
        **m_slide,
        "level": "slide",
        "n_test": len(ds_te),
    })

        fold_metrics.append({
            "fold": fold_name,
            **m_patient,
            "level": "patient",
            "n_test": len(patient_df),
        })
        # ====================================================

        # OG slide-level ============================================
        # m = train_one_fold(model, dl_tr, dl_te, mode) # OG

        # fold_metrics.append({"fold": fold_name, **m, "n_test": len(ds_te)})
        # print(f"[{fold_name}] auc={m['auc']:.3f} balacc={m['balacc']:.3f} acc={m['acc']:.3f} (n_test={len(ds_te)})")
        # OG ============================================

    # OG slide-level =========================================
    dfm = pd.DataFrame(fold_metrics)
    if len(dfm) == 0:
        print(f"No folds ran for {mode}")
        continue

    summary = summarize_percent(dfm, cols=("auc","balacc","acc"))
    all_rows.append({
        "experiment": mode,
        "AUC (mean±std %)": f"{summary['auc'][0]:.2f} ± {summary['auc'][1]:.2f}",
        "BAC (mean±std %)": f"{summary['balacc'][0]:.2f} ± {summary['balacc'][1]:.2f}",
        "ACC (mean±std %)": f"{summary['acc'][0]:.2f} ± {summary['acc'][1]:.2f}",
        "n_folds": len(dfm),
    })

    # OG slide-level =========================================

    dfm = pd.DataFrame(fold_metrics)



    if len(dfm) == 0:
        print(f"No folds ran for {mode}")
        continue

    # ----------------------------------
    # Slide-level summary
    # ----------------------------------
    slide_df = dfm[dfm["level"] == "slide"]

    slide_summary = summarize_percent(
        slide_df,
        cols=("auc", "balacc", "acc")
    )

    all_rows.append({
        "experiment": mode,
        "level": "slide",
        "AUC (mean±std %)": f"{slide_summary['auc'][0]:.2f} ± {slide_summary['auc'][1]:.2f}",
        "BAC (mean±std %)": f"{slide_summary['balacc'][0]:.2f} ± {slide_summary['balacc'][1]:.2f}",
        "ACC (mean±std %)": f"{slide_summary['acc'][0]:.2f} ± {slide_summary['acc'][1]:.2f}",
        "n_folds": len(slide_df),
    })

    # ----------------------------------
    # Patient-level summary
    # ----------------------------------
    patient_df = dfm[dfm["level"] == "patient"]

    patient_summary = summarize_percent(
        patient_df,
        cols=("auc", "balacc", "acc")
    )

    all_rows.append({
        "experiment": mode,
        "level": "patient",
        "AUC (mean±std %)": f"{patient_summary['auc'][0]:.2f} ± {patient_summary['auc'][1]:.2f}",
        "BAC (mean±std %)": f"{patient_summary['balacc'][0]:.2f} ± {patient_summary['balacc'][1]:.2f}",
        "ACC (mean±std %)": f"{patient_summary['acc'][0]:.2f} ± {patient_summary['acc'][1]:.2f}",
        "n_folds": len(patient_df),
    })

    # Optional debug print
    print(f"\n{mode}")
    print("Slide-level:")
    print(slide_df[["fold","auc","balacc","acc"]])

    print("\nPatient-level:")
    print(patient_df[["fold","auc","balacc","acc"]])

results_table = pd.DataFrame(all_rows)
print("\n==================== FINAL SUMMARY TABLE ====================")
print(results_table.to_string(index=False))



==================== vision_pool ====================

vision_pool
Slide-level:
        fold       auc    balacc       acc
0  FA_PT_k=0  0.780000  0.735455  0.744681
2  FA_PT_k=1  0.766957  0.720870  0.729167
4  FA_PT_k=2  0.774000  0.630000  0.600000
6  FA_PT_k=3  0.895000  0.667500  0.673469
8  FA_PT_k=4  0.747692  0.668462  0.666667

Patient-level:
        fold       auc    balacc       acc
1  FA_PT_k=0  0.711538  0.675481  0.689655
3  FA_PT_k=1  0.754808  0.706731  0.724138
5  FA_PT_k=2  0.764423  0.629808  0.655172
7  FA_PT_k=3  0.879808  0.661058  0.689655
9  FA_PT_k=4  0.774038  0.704327  0.689655

==================== vision_abmil ====================

vision_abmil
Slide-level:
        fold       auc    balacc       acc
0  FA_PT_k=0  0.763636  0.606364  0.595745
2  FA_PT_k=1  0.772174  0.702609  0.708333
4  FA_PT_k=2  0.754000  0.740000  0.733333
6  FA_PT_k=3  0.891667  0.794167  0.795918
8  FA_PT_k=4  0.750769  0.726923  0.725490

Patient-level:
        fold       auc    bala

: 